# Angriness Predictor v2 — 7 Behavioral Features, Semi-Supervised

This notebook documents the **v2 production pipeline** for the Angriness (Tilt) Predictor.

**Key change from v1:** Dropped all 15 non-behavioral features (ELO, time control, synthetic IoT data). The model now uses only **7 behavioral tilt features** — the same features the Isolation Forest was designed to detect tilt from. The synthetic IoT features (sleep, water, temperature, CO2, light) were randomly generated and added noise, not signal.

**Semi-supervised approach:**
1. **Stage 1 (IF):** Isolation Forest trained on 7 scaled behavioral features → generates pseudo-labels (angriness 1-5)
2. **Stage 2 (RF):** Random Forest Classifier trained on 7 **unscaled** behavioral features using IF labels → production model

RF is tree-based and scale-invariant, so training on unscaled features eliminates the need for a scaler at inference.

**Pipeline steps:**
1. **Gather** — Load and validate the training dataset
2. **Features** — Select 7 behavioral columns, scale for IF, boxplot quality check
3. **Train** — Stage 1: IF → pseudo-labels; Stage 2: RF on unscaled features
4. **Evaluate** — Accuracy/F1, proxy R², Spearman, overfitting diagnostics

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import pearsonr, spearmanr

TRAINER_DIR = Path.cwd().resolve().parent
RAW_DIR = TRAINER_DIR / "data" / "raw"
LEGACY_DIR = TRAINER_DIR / "data" / "legacy"
PLOT_DIR = Path.cwd() / "plots"
PLOT_DIR.mkdir(exist_ok=True)

print(f"Trainer root: {TRAINER_DIR}")
print(f"Raw data:     {RAW_DIR}")

## Step 1: Data Gathering

Load the training dataset and validate its structure. The dataset contains **25,249 games** from **~12,300 players** across balanced ELO brackets (800–3400+).

In [ ]:
DIVERSE_CSV = RAW_DIR / "angriness_dataset_diverse.csv"
ORIGINAL_CSV = LEGACY_DIR / "angriness_dataset.csv"

csv_path = DIVERSE_CSV if DIVERSE_CSV.exists() else ORIGINAL_CSV
if not csv_path.exists():
    raise FileNotFoundError(f"No dataset found. Expected {DIVERSE_CSV} or {ORIGINAL_CSV}")

df = pd.read_csv(csv_path)
print(f"Loaded: {csv_path.name}")
print(f"  Shape: {df.shape}")
print(f"  Unique players: {df['username'].nunique()}")
df.head()

In [ ]:
ANALYSIS_COLUMNS = [
    "acpl_player", "blunder_cnt_player", "mistake_cnt_player",
    "inaccuracy_cnt_player",
]

before = len(df)
df = df.dropna(subset=ANALYSIS_COLUMNS)
dropped = before - len(df)
print(f"Dropped {dropped} rows with null analysis -> {len(df)} remaining")

## Step 2: Feature Selection

Select only the **7 behavioral tilt features**. These are the features that directly measure in-game tilt behavior:

| Feature | Description |
|---|---|
| `consecutive_losses_pregame` | Loss streak before this game |
| `avg_tpm_seconds_player` | Average time per move (seconds) |
| `blunder_cnt_player` | Number of blunders |
| `mistake_cnt_player` | Number of mistakes |
| `inaccuracy_cnt_player` | Number of inaccuracies |
| `acpl_player` | Average centipawn loss |
| `accuracy_player` | Move accuracy percentage |

**Dropped (15 features):**
- ELO-related (4): elo, elo_diff, opponent_elo, elo_gap — empirically didn't improve model
- Time control (2): time_control_initial, time_control_increment — same
- Move counts (2): move_cnt, move_cnt_player
- Synthetic IoT (6): sleep_duration, awaken_duration, avg_ppm, avg_celsius, water_intake_ml, avg_lux — randomly generated, no real correlation with tilt
- Categorical (1): is_black

In [ ]:
BEHAVIORAL_FEATURES = [
    "consecutive_losses_pregame",
    "avg_tpm_seconds_player",
    "blunder_cnt_player",
    "mistake_cnt_player",
    "inaccuracy_cnt_player",
    "acpl_player",
    "accuracy_player",
]

df_raw = df[BEHAVIORAL_FEATURES].copy()

for col in df_raw.columns:
    if df_raw[col].isnull().any():
        df_raw[col] = df_raw[col].fillna(df_raw[col].median())

print(f"Selected {len(BEHAVIORAL_FEATURES)} behavioral features")
print(f"Shape: {df_raw.shape}")
print(f"Nulls: {df_raw.isnull().sum().sum()}")
df_raw.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6", "#1abc9c", "#e67e22"]

for i, col in enumerate(BEHAVIORAL_FEATURES):
    ax = axes[i]
    bp = ax.boxplot(df_raw[col].values, patch_artist=True, widths=0.6)
    bp["boxes"][0].set_facecolor(colors[i])
    bp["boxes"][0].set_alpha(0.7)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel("Value")
    median = df_raw[col].median()
    mean = df_raw[col].mean()
    ax.text(1.15, median, f"med={median:.1f}", va="center", fontsize=8, color="#2c3e50")
    ax.text(1.15, mean, f"avg={mean:.1f}", va="center", fontsize=8, color="#e74c3c")

axes[-1].set_visible(False)

fig.suptitle("Behavioral Feature Distributions (Unscaled)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOT_DIR / "behavioral_distributions.png", dpi=150)
plt.show()

In [ ]:
corr = df_raw.corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            annot_kws={"size": 9}, linewidths=0.5)
ax.set_title("Behavioral Feature Correlation Matrix")
fig.tight_layout()
fig.savefig(PLOT_DIR / "behavioral_correlation.png", dpi=150)
plt.show()

In [ ]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_raw),
    columns=BEHAVIORAL_FEATURES,
)

print(f"Scaled for IF: {df_scaled.shape}")
print(f"Note: Scaling is only needed for IF label generation.")
print(f"RF is trained on unscaled features and doesn't need scaling at inference.")

## Step 3: Training (Semi-Supervised)

### Stage 1: Isolation Forest (Label Generation)
IF trained on **scaled** 7 features → anomaly scores → percentile-binned into angriness 1-5.

### Stage 2: Random Forest Classifier
RF trained on **unscaled** 7 features using IF pseudo-labels. Since RF is tree-based, it's invariant to monotonic transformations — no scaler needed at inference.

**Data Split**: 64:16:20 (train:val:test) via two-stage 80:20.

In [ ]:
indices = np.arange(len(df_scaled))
trainval_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42)
train_idx, val_idx = train_test_split(trainval_idx, test_size=0.20, random_state=42)

# Scaled splits (for IF)
scaled_train = df_scaled.iloc[train_idx].reset_index(drop=True)
scaled_val = df_scaled.iloc[val_idx].reset_index(drop=True)
scaled_test = df_scaled.iloc[test_idx].reset_index(drop=True)

# Unscaled splits (for RF and evaluation)
raw_train = df_raw.iloc[train_idx].reset_index(drop=True)
raw_val = df_raw.iloc[val_idx].reset_index(drop=True)
raw_test = df_raw.iloc[test_idx].reset_index(drop=True)

print(f"Split: {len(scaled_train)} train / {len(scaled_val)} val / {len(scaled_test)} test ")
print(f"  ({len(scaled_train)/len(df_scaled):.0%}/{len(scaled_val)/len(df_scaled):.0%}/{len(scaled_test)/len(df_scaled):.0%})")

In [ ]:
# Stage 1: Isolation Forest
if_model = IsolationForest(
    contamination=0.03,
    n_estimators=200,
    max_features=0.75,
    random_state=42,
)

if_model.fit(scaled_train.values)

scores = if_model.decision_function(scaled_train.values)
n_anomalies = (if_model.predict(scaled_train.values) == -1).sum()
print(f"Stage 1: Isolation Forest trained on {len(BEHAVIORAL_FEATURES)} scaled features")
print(f"Anomalies: {n_anomalies} / {len(scaled_train)} ({n_anomalies / len(scaled_train):.1%})")

In [ ]:
PERCENTILE_EDGES = [0, 10, 35, 65, 90, 100]


def score_to_angriness(score, bin_edges):
    for i in range(len(bin_edges) - 1):
        if score <= bin_edges[i + 1]:
            return 5 - i
    return 1


bin_edges = [float(np.percentile(scores, p)) for p in PERCENTILE_EDGES]
print(f"Percentiles: {PERCENTILE_EDGES}")
print(f"Bin edges:   {[round(e, 4) for e in bin_edges]}")

y_train = np.array([score_to_angriness(s, bin_edges) for s in scores])

print(f"\nAngriness distribution (IF pseudo-labels):")
for level in range(1, 6):
    count = (y_train == level).sum()
    print(f"  Level {level}: {count:,} ({count / len(y_train):.1%})")

In [ ]:
# Stage 2: Random Forest on UNSCALED features
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(raw_train.values, y_train)

rf_pred_train = rf_model.predict(raw_train.values)
rf_accuracy = (rf_pred_train == y_train).mean()
print(f"Stage 2: Random Forest trained on {len(BEHAVIORAL_FEATURES)} unscaled features")
print(f"Train accuracy (vs IF labels): {rf_accuracy:.4f}")

In [ ]:
level_counts = pd.Series(rf_pred_train).value_counts().sort_index()
level_labels = {1: "Calm", 2: "Mild", 3: "Neutral", 4: "Annoyed", 5: "Tilted"}
level_colors = ["#2ecc71", "#a8e6cf", "#f9e79f", "#f5b041", "#e74c3c"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(level_counts.index, level_counts.values, color=level_colors, edgecolor="white", width=0.7)
for bar, count in zip(bars, level_counts.values):
    pct = count / len(rf_pred_train) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{count:,}\n({pct:.0f}%)", ha="center", fontsize=9)
ax.set_xticks(level_counts.index)
ax.set_xticklabels([f"{lvl} - {level_labels[lvl]}" for lvl in level_counts.index])
ax.set_xlabel("Angriness Level")
ax.set_ylabel("Number of Games")
ax.set_title("Angriness Level Distribution — RF Predictions (Training Set)")
fig.tight_layout()
fig.savefig(PLOT_DIR / "v2_angriness_distribution.png", dpi=150)
plt.show()

In [ ]:
tilt_cols = ["acpl_player", "blunder_cnt_player", "consecutive_losses_pregame"]
raw_check = raw_train.copy()
raw_check["_angriness"] = rf_pred_train

sanity = raw_check.groupby("_angriness")[tilt_cols].mean().round(2)
sanity.index.name = "Angriness"
sanity.columns = ["ACPL", "Blunders", "Consecutive Losses"]
print("Sanity check — unscaled feature means by angriness level (train set):\n")
sanity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (col, label) in zip(axes, [("acpl_player", "ACPL"), ("blunder_cnt_player", "Blunders"), ("consecutive_losses_pregame", "Consecutive Losses")]):
    means = raw_check.groupby("_angriness")[col].mean()
    bars = ax.bar(means.index, means.values, color=level_colors, edgecolor="white", width=0.7)
    ax.set_xlabel("Angriness Level")
    ax.set_ylabel(f"Mean {label}")
    ax.set_title(label)
    ax.set_xticks(means.index)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("Tilt Features by Angriness Level (RF, Unscaled)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOT_DIR / "v2_sanity_check_by_level.png", dpi=150)
plt.show()

## Step 4: Evaluation

Evaluate the semi-supervised RF model on held-out test data:
- **Accuracy / F1** — how well RF reproduces IF labels on unseen data
- **Proxy R²** — correlation between predicted class and composite tilt signal
- **Spearman ρ** — rank correlation with individual tilt features
- **Validation checks** — tilted games (4-5) should have higher ACPL, more blunders, more losses than calm (1-2)

In [ ]:
rf_pred_test = rf_model.predict(raw_test.values)

# IF labels on test set for accuracy
if_scores_test = if_model.decision_function(scaled_test.values)
y_test_if = np.array([score_to_angriness(s, bin_edges) for s in if_scores_test])
test_acc = accuracy_score(y_test_if, rf_pred_test)
test_f1 = f1_score(y_test_if, rf_pred_test, average="weighted")
print(f"Test accuracy (vs IF labels): {test_acc:.4f}")
print(f"Test F1 (weighted):           {test_f1:.4f}")

raw = raw_test.copy()
raw["_angriness"] = rf_pred_test

per_level = {}
for level in range(1, 6):
    mask = raw["_angriness"] == level
    subset = raw.loc[mask]
    per_level[str(level)] = {
        "count": int(mask.sum()),
        "mean_acpl": round(float(subset["acpl_player"].mean()), 2) if len(subset) > 0 else None,
        "mean_blunders": round(float(subset["blunder_cnt_player"].mean()), 2) if len(subset) > 0 else None,
        "mean_cons_losses": round(float(subset["consecutive_losses_pregame"].mean()), 2) if len(subset) > 0 else None,
    }

print(f"\nTest set: {len(raw_test)} games\n")
pd.DataFrame(per_level).T

In [ ]:
calm = raw[raw["_angriness"].isin([1, 2])]
tilted = raw[raw["_angriness"].isin([4, 5])]

checks = {
    "ACPL higher when tilted": tilted["acpl_player"].mean() > calm["acpl_player"].mean(),
    "Blunders higher when tilted": tilted["blunder_cnt_player"].mean() > calm["blunder_cnt_player"].mean(),
    "Consecutive losses higher when tilted": tilted["consecutive_losses_pregame"].mean() > calm["consecutive_losses_pregame"].mean(),
}

all_pass = all(checks.values())
print(f"Validation: {'PASS' if all_pass else 'FAIL'}\n")
for check, result in checks.items():
    print(f"  {'PASS' if result else 'FAIL'} — {check}")

In [ ]:
compare_cols = ["acpl_player", "blunder_cnt_player", "consecutive_losses_pregame"]
compare_labels = ["ACPL", "Blunders", "Consecutive\nLosses"]

calm_means = calm[compare_cols].mean()
tilted_means = tilted[compare_cols].mean()

x = np.arange(len(compare_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, calm_means.values, width, label="Calm (1-2)", color="#2ecc71", edgecolor="white")
bars2 = ax.bar(x + width / 2, tilted_means.values, width, label="Tilted (4-5)", color="#e74c3c", edgecolor="white")

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(compare_labels)
ax.set_ylabel("Mean Value")
ax.set_title("Calm vs Tilted — Key Tilt Features (Test Set)")
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "v2_calm_vs_tilted.png", dpi=150)
plt.show()

### Overfitting Diagnostics: Train vs Val vs Test

- **Accuracy / F1** — RF achieves ~100% on train (by design); val vs test gap indicates generalization
- **Proxy R²** — Pearson r² between predicted class and composite tilt
- **Spearman ρ** — Rank correlation with individual tilt features

In [ ]:
COMPOSITE_COLS = ["acpl_player", "blunder_cnt_player", "consecutive_losses_pregame"]
SHORT_NAMES = {"acpl_player": "ACPL", "blunder_cnt_player": "Blunders", "consecutive_losses_pregame": "ConsLoss"}


def compute_composite_tilt(raw_df):
    components = []
    for c in COMPOSITE_COLS:
        vals = raw_df[c].values.astype(float)
        vmin, vmax = vals.min(), vals.max()
        components.append((vals - vmin) / (vmax - vmin + 1e-9))
    return np.mean(components, axis=0)


def evaluate_split(name, raw_df, scaled_df, rf_mdl, if_mdl, edges):
    pred = rf_mdl.predict(raw_df.values)
    if_scores = if_mdl.decision_function(scaled_df.values)
    y_if = np.array([score_to_angriness(s, edges) for s in if_scores])
    acc = round(float(accuracy_score(y_if, pred)), 4)
    f1 = round(float(f1_score(y_if, pred, average="weighted")), 4)

    composite = compute_composite_tilt(raw_df)
    r, _ = pearsonr(pred.astype(float), composite)
    proxy_r2 = round(float(r ** 2), 4)

    spearman = {}
    for col in COMPOSITE_COLS:
        corr, _ = spearmanr(pred.astype(float), raw_df[col].values)
        spearman[col] = round(float(corr), 4)

    return {"name": name, "n": len(raw_df), "accuracy": acc, "f1": f1, "proxy_r2": proxy_r2, "spearman": spearman}


diag_results = []
for name, raw_split, scaled_split in [
    ("train", raw_train, scaled_train),
    ("val", raw_val, scaled_val),
    ("test", raw_test, scaled_test),
]:
    diag_results.append(evaluate_split(name, raw_split, scaled_split, rf_model, if_model, bin_edges))

print("Diagnostics computed for all 3 splits.")

In [ ]:
R2_GAP_THRESHOLD = 0.05

print("=== Train vs Val vs Test Comparison ===\n")
header = f"{'Split':>6}  {'N':>6}  {'Accuracy':>8}  {'F1':>6}  {'Proxy R2':>8}"
for col in COMPOSITE_COLS:
    header += f"  {SHORT_NAMES[col] + ' rho':>12}"
print(header)
print("-" * len(header))

for r in diag_results:
    row = f"{r['name']:>6}  {r['n']:>6}  {r['accuracy']:>8.4f}  {r['f1']:>6.4f}  {r['proxy_r2']:>8.4f}"
    for col in COMPOSITE_COLS:
        row += f"  {r['spearman'][col]:>12.4f}"
    print(row)

val_r = next(r for r in diag_results if r['name'] == 'val')
test_r = next(r for r in diag_results if r['name'] == 'test')
train_r = next(r for r in diag_results if r['name'] == 'train')
r2_gap = train_r['proxy_r2'] - test_r['proxy_r2']
acc_gap = val_r['accuracy'] - test_r['accuracy']

issues = []
if r2_gap > R2_GAP_THRESHOLD:
    issues.append(f"R2 gap = {r2_gap:.4f}")
if abs(acc_gap) > R2_GAP_THRESHOLD:
    issues.append(f"Accuracy gap (val vs test) = {acc_gap:.4f}")

print(f"\n{'=' * 50}")
if issues:
    print("POSSIBLE OVERFITTING DETECTED:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("NO OVERFITTING DETECTED")
    print(f"  R2 gap:                     {r2_gap:.4f} (threshold: {R2_GAP_THRESHOLD})")
    print(f"  Accuracy gap (val vs test): {acc_gap:.4f} (threshold: {R2_GAP_THRESHOLD})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
split_names = [r["name"] for r in diag_results]
colors_split = ["#3498db", "#f39c12", "#e74c3c"]

r2_values = [r["proxy_r2"] for r in diag_results]
axes[0].bar(split_names, r2_values, color=colors_split, edgecolor="white")
axes[0].set_ylabel("Proxy R\u00b2")
axes[0].set_title("Proxy R\u00b2 by Split")
for i, v in enumerate(r2_values):
    axes[0].text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=10)

x = np.arange(len(COMPOSITE_COLS))
width = 0.25
for i, r in enumerate(diag_results):
    rhos = [r["spearman"][col] for col in COMPOSITE_COLS]
    axes[1].bar(x + i * width, rhos, width, label=r["name"], color=colors_split[i], edgecolor="white")
axes[1].set_xticks(x + width)
axes[1].set_xticklabels([SHORT_NAMES[c] for c in COMPOSITE_COLS])
axes[1].set_ylabel("Spearman rho")
axes[1].set_title("Spearman Correlation by Feature and Split")
axes[1].legend()

fig.suptitle("Overfitting Diagnostics (RF v2 — 7 Behavioral Features)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOT_DIR / "v2_overfitting_diagnostics.png", dpi=150)
plt.show()

In [ ]:
print(f"Pipeline v2 complete. All validations {'PASSED' if all_pass else 'FAILED'}.")
print(f"\nModel: Semi-Supervised (IF + RF), 7 behavioral features, unscaled")
print(f"Split: {len(scaled_train)} train / {len(scaled_val)} val / {len(scaled_test)} test")
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test F1 (weighted): {test_f1:.4f}")
print(f"Proxy R\u00b2: train={diag_results[0]['proxy_r2']:.4f} / val={diag_results[1]['proxy_r2']:.4f} / test={diag_results[2]['proxy_r2']:.4f}")
print(f"Overfitting: {'NOT DETECTED' if not issues else 'DETECTED'}")
print(f"\nTo generate production artifacts: python run_pipeline.py")